# Distilling a 0.8B Retail Tool-Calling Agent

In this notebook, we are going to take a tiny model and ask a very concrete question:

> Can `mlx-community/Qwen3.5-0.8B-MLX-bf16` become more reliable at one focused multi-turn retail-support task after sequence-level distillation from a stronger teacher?

This is not a general chat-model benchmark. We are treating the small model as an action policy inside a tool loop. It sees a customer-support conversation and available retail tools, emits a tool call or a user-facing response, receives the tool result or next user message, and keeps going until the task is done.


## The Shape Of The Experiment

We will move through one full loop across this blog:

1. Load the τ³-Bench retail domain: tasks, policy, tools, database, and official train/test split.
2. Run the base 0.8B student model and inspect where it fails.
3. Ask a stronger teacher model to produce successful retail-support trajectories.
4. Fine-tune the student on those teacher trajectories using normal next-token training.
5. Run the same held-out retail tasks again and compare the behavior.

The important concept is that a benchmark is not just a dataset. For tool use, the benchmark is a contract: data, prompt format, parser, environment, user simulator, and scorer all have to agree.


## Naming Convention

The public benchmark family is **τ-Bench**. This blog uses the current **τ³-Bench** retail domain. The implementation still lives in the GitHub repository `sierra-research/tau2-bench`, and the Python package, CLI, and data paths are named `tau2`.

So in this notebook:

- **τ³-Bench retail** means the benchmark/release/domain we evaluate on.
- `tau2-bench`, `tau2`, and `data/tau2/...` mean upstream repository, Python package, CLI, or file paths.

That split is annoying, but it keeps the prose aligned with the benchmark branding and the code aligned with the actual package names.


## What Distillation Means Here

In this first blog, distillation means **offline sequence-level imitation**. We first let a stronger teacher solve train tasks inside the same tool loop. We keep the successful teacher trajectories, then turn each teacher action into a supervised training row:

```text
input  = conversation history and tool observations before the teacher action
target = the exact teacher action text
```

The student is trained with the normal next-token loss to reproduce those target tokens. So yes, the loss is still token-level, but the teacher signal is the chosen text sequence, not the teacher's probability distribution.

That means this blog is **not** logit distillation yet, and it is **not** RL yet. Logit distillation comes later, when we ask the teacher what probabilities it assigned to tokens. RL comes later, when we update the student from rewards produced by the simulator/evaluator.


## What We Are Not Doing Yet

This first notebook keeps the harness visible. We will use plain Python before bringing in orchestration frameworks like LangGraph or structured-output helpers like BAML.

That is intentional. If the small model emits an invalid tool call, we want to see the invalid tool call. Later notebooks can ask how engineering scaffolding changes reliability. This notebook asks what the model learns from distillation.


In [1]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from typing import Any

import json

from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import qwen_format
from common import retail_eval
from common import tau_runtime
from common import trace_stats
from common import user_simulator

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

STUDENT_MODEL = cfg.STUDENT_MODEL
TAU_BENCH_REPO_REVISION = cfg.TAU_BENCH_REPO_REVISION
TAU_BENCH_REPO = "sierra-research/tau2-bench"
TAU_BENCH_DOMAIN = "retail"
MAX_NEW_TOKENS = cfg.MAX_NEW_TOKENS

print(f"Project root: {ROOT}")
print(f"Student model: {STUDENT_MODEL}")
print(f"Benchmark: τ³-Bench {TAU_BENCH_DOMAIN}")
print(f"Runtime repo/package: {TAU_BENCH_REPO} / tau2")
print(f"Pinned revision: {TAU_BENCH_REPO_REVISION[:8]}")
print(f"Loaded .env: {DOTENV_LOADED} from {ENV_PATH}")

Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Student model: mlx-community/Qwen3.5-0.8B-MLX-bf16
Benchmark: τ³-Bench retail
Runtime repo/package: sierra-research/tau2-bench / tau2
Pinned revision: c42db6cc
Loaded .env: True from /Users/kargarisaac/codes/personal/distillation-blogs/.env


## First Mental Model: The Harness

A normal text benchmark can look like this:

`question -> model -> final answer -> string comparison`

A tool-use benchmark looks more like this:

`state -> prompt -> model action -> parse action -> execute tool -> new state -> repeat -> score`

That loop is the harness. The model is only one part of the system. The environment and scorer are what make the task measurable.


## Two Views Of The Same Loop

There are two useful ways to look at the system.

The engineering view separates the pieces so we can debug them. The model proposes actions. The harness renders prompts, parses tool calls, controls the loop, and logs what happened. The simulator owns the hidden world state and returns observations when tools are executed. The scorer turns the outcome into metrics or reward.

![Engineering view of model, harness, simulator, and scorer boundaries](assets/harness-boundaries.png)


The RL view wraps most of those pieces into one larger object: the environment. From that angle, the policy is the model, and everything outside the model is the environment. The model emits an action, the environment returns an observation and reward, and the optimizer uses that signal to update the model.

Both views are correct. The engineering view is better for building and debugging. The RL view is better for thinking about training.

![RL view of policy, environment, reward, and optimizer loop](assets/rl-environment-loop.png)


## A Tiny Tool-Use Benchmark

Before we touch τ³-Bench retail, we will build the smallest possible version of the same idea.

The task is: create a file with a specific path and content. The model's action is a structured tool call. The simulator is a fake file system. The scorer checks the final file-system state.

This toy benchmark is not the research result. It is here to make the moving parts visible.


In [2]:
@dataclass
class ToyTask:
    user_request: str
    expected_path: str
    expected_content: str


@dataclass
class ToolResult:
    ok: bool
    observation: str


class ToyFileSystem:
    def __init__(self) -> None:
        self.files: dict[str, str] = {}

    def write_file(self, path: str, content: str) -> ToolResult:
        self.files[path] = content
        return ToolResult(ok=True, observation=f"Wrote {path}")

    def snapshot(self) -> dict[str, str]:
        return dict(self.files)


toy_task = ToyTask(
    user_request="Create a note called plan.txt with the content hello from the harness.",
    expected_path="plan.txt",
    expected_content="hello from the harness",
)

toy_env = ToyFileSystem()
toy_env.snapshot()


{}

The important boundary is that the model does not directly mutate the file system. It only proposes an action. The harness parses that action, decides whether it is valid, executes it, and then scores what happened.


In [3]:
def render_toy_prompt(task: ToyTask, env: ToyFileSystem) -> str:
    tools = [
        {
            "name": "write_file",
            "description": "Write text content to a file path.",
            "parameters": {
                "path": "string",
                "content": "string",
            },
        }
    ]
    return json.dumps(
        {
            "user_request": task.user_request,
            "available_tools": tools,
            "current_files": env.snapshot(),
            "instruction": "Return exactly one tool call as JSON with name and arguments.",
        },
        indent=2,
    )


def parse_json_tool_call(raw_text: str) -> tuple[dict[str, Any] | None, list[str]]:
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        return None, [f"Invalid JSON: {exc}"]

    errors: list[str] = []
    if not isinstance(parsed, dict):
        errors.append("Tool call must be a JSON object.")
    if parsed.get("name") != "write_file":
        errors.append("Tool call name must be write_file.")

    arguments = parsed.get("arguments")
    if not isinstance(arguments, dict):
        errors.append("Tool call arguments must be an object.")
    else:
        if not isinstance(arguments.get("path"), str):
            errors.append("arguments.path must be a string.")
        if not isinstance(arguments.get("content"), str):
            errors.append("arguments.content must be a string.")

    return parsed if not errors else None, errors


def execute_json_tool_call(env: ToyFileSystem, tool_call: dict[str, Any]) -> ToolResult:
    arguments = tool_call["arguments"]
    return env.write_file(path=arguments["path"], content=arguments["content"])


def score_toy_task(task: ToyTask, env: ToyFileSystem) -> dict[str, Any]:
    actual_content = env.files.get(task.expected_path)
    return {
        "success": actual_content == task.expected_content,
        "expected_path": task.expected_path,
        "expected_content": task.expected_content,
        "actual_content": actual_content,
        "final_files": env.snapshot(),
    }


Now we can run the full harness with a fake model. The fake model is just a Python function for now. Later, this exact slot becomes `mlx-community/Qwen3.5-0.8B-MLX-bf16`.


In [4]:
def good_fake_model(prompt: str) -> str:
    return json.dumps(
        {
            "name": "write_file",
            "arguments": {
                "path": "plan.txt",
                "content": "hello from the harness",
            },
        }
    )


def bad_fake_model(prompt: str) -> str:
    return "Done, I created the note."


def run_toy_harness(task: ToyTask, model_fn) -> dict[str, Any]:
    env = ToyFileSystem()
    prompt = render_toy_prompt(task, env)
    raw_model_output = model_fn(prompt)
    tool_call, parse_errors = parse_json_tool_call(raw_model_output)

    tool_result = None
    if tool_call is not None:
        tool_result = execute_json_tool_call(env, tool_call)

    return {
        "prompt": prompt,
        "raw_model_output": raw_model_output,
        "parse_errors": parse_errors,
        "tool_result": tool_result,
        "score": score_toy_task(task, env),
    }


good_run = run_toy_harness(toy_task, good_fake_model)
bad_run = run_toy_harness(toy_task, bad_fake_model)

print("Good fake model score:")
print(json.dumps(good_run["score"], indent=2))
print("\nBad fake model parse errors:")
print(json.dumps(bad_run["parse_errors"], indent=2))
print("\nBad fake model score:")
print(json.dumps(bad_run["score"], indent=2))


Good fake model score:
{
  "success": true,
  "expected_path": "plan.txt",
  "expected_content": "hello from the harness",
  "actual_content": "hello from the harness",
  "final_files": {
    "plan.txt": "hello from the harness"
  }
}

Bad fake model parse errors:
[
  "Invalid JSON: Expecting value: line 1 column 1 (char 0)"
]

Bad fake model score:
{
  "success": false,
  "expected_path": "plan.txt",
  "expected_content": "hello from the harness",
  "actual_content": null,
  "final_files": {}
}


## Now Swap In τ³-Bench Retail

The old τ-Bench repo now warns that its airline and retail tasks are outdated. For this blog, we use the newer `sierra-research/tau2-bench` repository, which is the current τ³-Bench code/data release and includes fixed retail tasks.

We are choosing only the retail domain because the student is tiny. The goal is not to make a universal agent yet. The goal is to test whether a 0.8B model can specialize into one realistic customer-support workflow.

A retail task has more moving parts than the toy file-system example:

- `policy.md`: the rules the support agent must follow.
- `tools.py`: the available retail API actions.
- `db.json`: the hidden store database that tools read and mutate.
- `tasks.json`: hidden user scenarios and evaluation criteria.
- `split_tasks.json`: official train/test task split.


In [5]:
retail_domain = tau_runtime.load_tau_bench_retail_domain(DATA_DIR, TAU_BENCH_REPO_REVISION)
tau_bench_runtime = retail_domain.runtime
TAU_BENCH_RETAIL_DATA_DIR = retail_domain.data_dir
TAU_BENCH_RETAIL_SOURCE_DIR = retail_domain.source_dir
tau_bench_paths = retail_domain.paths
retail_environment = retail_domain.environment
retail_policy = retail_domain.policy

print("Pinned tau2-bench checkout:", tau_bench_runtime.source_checkout)
print()
print("τ³-Bench retail files:")
for name, path in tau_bench_paths.items():
    print(f"{name:12s} {path} ({path.stat().st_size:,} bytes)")


Pinned tau2-bench checkout: /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench

τ³-Bench retail files:
tasks        /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench/data/tau2/domains/retail/tasks.json (345,982 bytes)
splits       /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench/data/tau2/domains/retail/split_tasks.json (3,263 bytes)
policy       /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench/data/tau2/domains/retail/policy.md (6,699 bytes)
db           /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench/data/tau2/domains/retail/db.json (2,811,616 bytes)


The retail files are normal JSON/Markdown/Python source files. We inspect those files first, then use the official `tau2` runtime for the real benchmark loop: user simulation, environment state, tool execution, and reward calculation.


The τ³-Bench retail benchmark is a small world, not just a list of prompts. The student model is the agent. The user simulator plays the customer from hidden task instructions. The harness controls the conversation and tool execution. The retail environment owns the hidden database. The scorer checks whether the final state matches the task goal.

![τ³-Bench retail loop: main LLM agent, user simulator, harness, retail environment, and reward](assets/tau3-bench-retail-loop.png)


In [6]:
retail_tasks = json.loads(tau_bench_paths["tasks"].read_text())
retail_splits = json.loads(tau_bench_paths["splits"].read_text())
retail_policy = tau_bench_paths["policy"].read_text()
retail_db = json.loads(tau_bench_paths["db"].read_text())

retail_tasks_by_id = {task["id"]: task for task in retail_tasks}
retail_train_tasks = [retail_tasks_by_id[task_id] for task_id in retail_splits["train"]]
retail_eval_tasks = [retail_tasks_by_id[task_id] for task_id in retail_splits["test"]]

reward_basis_counts = Counter(
    tuple(task["evaluation_criteria"]["reward_basis"])
    for task in retail_tasks
)
action_count_distribution = Counter(
    len(task["evaluation_criteria"]["actions"])
    for task in retail_tasks
)

print("Retail task count:", len(retail_tasks))
print("Train split:", len(retail_train_tasks))
print("Held-out test split:", len(retail_eval_tasks))
print("Base split:", len(retail_splits["base"]))
print("\nReward basis counts:")
for reward_basis, count in reward_basis_counts.items():
    print(f"  {list(reward_basis)}: {count}")
print("\nReference action count distribution:")
for action_count, count in sorted(action_count_distribution.items()):
    print(f"  {action_count:2d} actions: {count} tasks")


Retail task count: 114
Train split: 74
Held-out test split: 40
Base split: 114

Reward basis counts:
  ['DB', 'NL_ASSERTION']: 112
  ['DB']: 2

Reference action count distribution:
   0 actions: 2 tasks
   1 actions: 20 tasks
   2 actions: 15 tasks
   3 actions: 9 tasks
   4 actions: 8 tasks
   5 actions: 18 tasks
   6 actions: 16 tasks
   7 actions: 8 tasks
   8 actions: 2 tasks
   9 actions: 1 tasks
  10 actions: 4 tasks
  11 actions: 2 tasks
  12 actions: 5 tasks
  13 actions: 4 tasks


Notice the split size: retail has **74 train tasks** and **40 held-out test tasks**. We should use the official held-out size so the experiment stays aligned with the benchmark.


In [7]:
first_train_task = retail_train_tasks[0]
first_eval_task = retail_eval_tasks[0]


def show_retail_task(task: dict[str, Any]) -> None:
    instructions = task["user_scenario"]["instructions"]
    criteria = task["evaluation_criteria"]
    print("Task id:", task["id"])
    print("\nHidden user-simulator instructions:")
    for key in ["task_instructions", "reason_for_call", "known_info", "unknown_info"]:
        print(f"- {key}: {instructions.get(key)}")
    print("\nReward basis:", criteria["reward_basis"])
    print("Reference trajectory length:", len(criteria["actions"]))
    print("\nFirst few reference actions:")
    for action in criteria["actions"]:
        print(json.dumps(action, indent=2))


show_retail_task(first_train_task)


Task id: 0

Hidden user-simulator instructions:
- task_instructions: You are detail-oriented and want to make sure everything is addressed in one go.
- reason_for_call: You received your order #W2378156 and wish to exchange the mechanical keyboard for the same one but with clicky switches and the smart thermostat for one compatible with Google Home instead of Apple HomeKit. If there is no keyboard that is clicky, RGB backlight, full size, you'd go for no backlight.
- known_info: You are Yusuf Rossi in zip code 19122.
- unknown_info: You do not remember your email address.

Reward basis: ['DB', 'NL_ASSERTION']
Reference trajectory length: 5

First few reference actions:
{
  "action_id": "0_0",
  "name": "find_user_id_by_name_zip",
  "arguments": {
    "first_name": "Yusuf",
    "last_name": "Rossi",
    "zip": "19122"
  },
  "info": null
}
{
  "action_id": "0_1",
  "name": "get_order_details",
  "arguments": {
    "order_id": "#W2378156"
  },
  "info": null
}
{
  "action_id": "0_2",
  "

This is the most important τ³-Bench idea to keep straight:

The `evaluation_criteria.actions` list is **not** the only valid path the agent may take. It is one reference trajectory. The official evaluator can replay those actions on a fresh environment to derive the target database state, then compare the agent's final database state against that target.

So for training, a teacher trajectory is useful because it gives the student one good way to solve the task. For evaluation, we care about outcome: did the conversation and tool use produce the right final state while following policy?


In [8]:
def summarize_retail_db(db: dict[str, Any]) -> dict[str, Any]:
    orders = db.get("orders", {})
    users = db.get("users", {})
    products = db.get("products", {})
    status_counts = Counter(order.get("status") for order in orders.values())
    return {
        "products": len(products),
        "users": len(users),
        "orders": len(orders),
        "order_status_counts": dict(status_counts),
    }


print(json.dumps(summarize_retail_db(retail_db), indent=2))


{
  "products": 50,
  "users": 500,
  "orders": 1000,
  "order_status_counts": {
    "processed": 102,
    "delivered": 373,
    "pending": 423,
    "cancelled": 102
  }
}


The model should not see this whole database in its prompt. The database is hidden environment state. The agent learns about it by calling tools like `find_user_id_by_name_zip`, `get_order_details`, and `get_product_details`.

That is exactly why this is a good first real task for us: the world is still realistic and stateful, but it is focused on one domain.


## Load The Retail Tool Definitions

τ³-Bench defines retail tools as Python methods, and the runtime exposes those tools through `environment.get_tools()`. We use that official metadata, especially each tool's `openai_schema`, then convert it into the simple `{"name", "description", "parameters"}` shape that Qwen's tokenizer expects.

So there is no custom source-code parser here. The notebook only adapts official τ³-Bench tool metadata for prompt rendering; official scoring still uses the τ³-Bench environment because it owns the real tool implementations, user simulator, and reward calculation.


In [9]:
retail_tools = retail_domain.tools
retail_tool_schema_by_name = retail_domain.tool_schema_by_name
retail_tool_names = [tool["name"] for tool in retail_tools]

print("Retail tools:", len(retail_tools))
print(retail_tool_names)
print()
print("First tool schema:")
print(json.dumps(retail_tools[0], indent=2)[:1200])


Retail tools: 16
['calculate', 'cancel_pending_order', 'exchange_delivered_order_items', 'find_user_id_by_name_zip', 'find_user_id_by_email', 'get_order_details', 'get_product_details', 'get_item_details', 'get_user_details', 'list_all_product_types', 'modify_pending_order_address', 'modify_pending_order_items', 'modify_pending_order_payment', 'modify_user_address', 'return_delivered_order_items', 'transfer_to_human_agents']

First tool schema:
{
  "name": "calculate",
  "description": "Calculate the result of a mathematical expression.",
  "parameters": {
    "properties": {
      "expression": {
        "description": "The mathematical expression to calculate, such as '2 + 2'. The expression can contain numbers, operators (+, -, *, /), parentheses, and spaces.",
        "title": "Expression",
        "type": "string"
      }
    },
    "required": [
      "expression"
    ],
    "title": "parameters",
    "type": "object"
  }
}


## Render A Qwen Prompt For One Retail Probe

In official τ³-Bench evaluation, the agent does **not** receive the hidden task instructions directly. A user simulator sees those instructions and talks to the agent over multiple turns.

For this first notebook probe, we will create a visible debug user message from the task scenario so we can inspect the Qwen chat template and the student's first action. This is a prompt-format probe, not an official benchmark score.


In [10]:
from mlx_lm import generate as mlx_generate
from mlx_lm import load as mlx_load
from mlx_lm.sample_utils import make_sampler


def debug_user_message_from_task(task: dict[str, Any]) -> str:
    instructions = task["user_scenario"]["instructions"]
    return "\n".join(
        part
        for part in [
            instructions.get("reason_for_call"),
            instructions.get("known_info"),
            instructions.get("unknown_info"),
        ]
        if part
    )


# Load the MLX model/tokenizer once and reuse the tokenizer for prompt rendering.
# We do not generate yet; the next cells first inspect the rendered prompt.
student_model, tokenizer = mlx_load(STUDENT_MODEL)
student_sampler = make_sampler(temp=0.0, top_p=1.0, top_k=0)

retail_system_message = tau_runtime.retail_agent_system_prompt(retail_policy)
retail_probe_messages = [
    {"role": "system", "content": retail_system_message},
    {"role": "user", "content": debug_user_message_from_task(first_train_task)},
]

retail_first_turn_prompt = tokenizer.apply_chat_template(
    retail_probe_messages,
    tools=retail_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("Loaded student with MLX-LM:", STUDENT_MODEL)
print(retail_first_turn_prompt[:4000])


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loaded student with MLX-LM: mlx-community/Qwen3.5-0.8B-MLX-bf16
<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"name": "calculate", "description": "Calculate the result of a mathematical expression.", "parameters": {"properties": {"expression": {"description": "The mathematical expression to calculate, such as '2 + 2'. The expression can contain numbers, operators (+, -, *, /), parentheses, and spaces.", "title": "Expression", "type": "string"}}, "required": ["expression"], "title": "parameters", "type": "object"}}
{"name": "cancel_pending_order", "description": "Cancel a pending order. If the order is already processed or delivered,\n\nit cannot be cancelled. The agent needs to explain the cancellation detail\nand ask for explicit user confirmation (yes/no) to proceed. If the user confirms,\nthe order status will be changed to 'cancelled' and the payment will be refunded.\nThe refund will be added to the user's gift card balance immediately if the pa

Pause here before running the model. The prompt should show three layers:

1. The retail policy in the system message.
2. The tool schemas inside Qwen's `<tools>...</tools>` section.
3. The customer message asking for help.

If any of those layers are missing, the model is being evaluated under the wrong contract.


## Run The Base Student Once

Now we ask the base 0.8B model for one continuation. This is not the full benchmark loop yet. We are checking whether the model can read a retail-support prompt and emit a plausible next action in Qwen's native tool-call format.


In [11]:
if "student_model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run the prompt-rendering cell first; it loads the MLX model and tokenizer.")
if "student_sampler" not in globals():
    raise RuntimeError("Run the prompt-rendering cell first; it creates the MLX sampler.")

print("Student ready for MLX-LM generation:", STUDENT_MODEL)


Student ready for MLX-LM generation: mlx-community/Qwen3.5-0.8B-MLX-bf16


In [12]:
MAX_NEW_TOKENS = 2048

raw_student_output = mlx_generate(
    student_model,
    tokenizer,
    prompt=retail_first_turn_prompt,
    max_tokens=MAX_NEW_TOKENS,
    sampler=student_sampler,
    verbose=False,
)

print("Raw student output:")
print(raw_student_output)


Raw student output:
I need to help you with your order exchange. First, I need to authenticate your identity. You mentioned you're Yusuf Rossi in zip code 19122, but I don't remember your email address. Let me try to find your user ID using your name and zip code.

<tool_call>
<function=find_user_id_by_name_zip>
<parameter=first_name>
Yusuf
</parameter>
<parameter=last_name>
Rossi
</parameter>
<parameter=zip>
19122
</parameter>
</function>
</tool_call>


After running this, inspect the output before moving on. We care about questions like:

- Did it call a retail tool or just chat?
- If it called a tool, did it use Qwen's `<tool_call>` format?
- Did it pick a reasonable first tool, such as authenticating the user or looking up the order?
- Did it try to mutate state before asking for required confirmation?

Those are behavior questions. The score comes later, when we wire the official τ³-Bench user simulator and evaluator.


## Parse Qwen Tool Calls

The parser boundary is still useful after the dataset swap. Qwen's chat template asks for XML-like tool calls, and our harness needs to turn those strings into structured function calls.


In [13]:
if "raw_student_output" not in globals():
    raise RuntimeError("Run the student generation cell first, so raw_student_output exists.")

parse_result = qwen_format.parse_qwen_tool_calls(raw_student_output)
print("Parse errors:", parse_result.errors)
print("Parsed calls:")
for call in parse_result.calls:
    print(call)


Parse errors: []
Parsed calls:
ParsedQwenToolCall(name='find_user_id_by_name_zip', arguments={'first_name': 'Yusuf', 'last_name': 'Rossi', 'zip': '19122'}, raw_block='<function=find_user_id_by_name_zip>\n<parameter=first_name>\nYusuf\n</parameter>\n<parameter=last_name>\nRossi\n</parameter>\n<parameter=zip>\n19122\n</parameter>\n</function>')


## Step Through One Retail Agent Run

Before running the probe or official eval, run one retail task through the loop and print each boundary clearly. This is a teaching/debug run, not the official τ³-Bench score: we use one visible debug user message from the task and one optional scripted confirmation so we can watch the student interact with the real retail tool environment.


In [14]:
# Teaching/debug multi-turn run for one retail task.
# This is not official τ³-Bench scoring. The official benchmark uses an LLM user
# simulator and evaluator; here we make the loop visible for one task.
import textwrap
from typing import Any

DEBUG_RETAIL_TASK = first_train_task
DEBUG_AGENT_MAX_STEPS = 10
DEBUG_USER_FOLLOW_UPS = [
    "Yes, I confirm. Please proceed with the exchange.",
]


def print_turn(label: str, text: Any) -> None:
    print("\n" + "=" * 96)
    print(label)
    print("-" * 96)
    if isinstance(text, (dict, list)):
        text = json.dumps(cfg.make_json_safe(text), indent=2, ensure_ascii=False)
    text = str(text)
    if "\n" in text:
        print(text)
    else:
        print(textwrap.fill(text, width=110, replace_whitespace=False))


def render_retail_debug_prompt(messages: list[dict[str, str]]) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": retail_system_message}, *messages],
        tools=retail_tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def generate_student_debug_turn(messages: list[dict[str, str]]) -> str:
    prompt = render_retail_debug_prompt(messages)
    return mlx_generate(
        student_model,
        tokenizer,
        prompt=prompt,
        max_tokens=MAX_NEW_TOKENS,
        sampler=student_sampler,
        verbose=False,
    )


if "student_model" not in globals():
    raise RuntimeError("Run the MLX student model loading cell first.")

retail_tool_schema_by_name = {tool["name"]: tool for tool in retail_tools}
tau_bench_runtime = tau_runtime.load_tau_bench_retail_runtime(DATA_DIR, TAU_BENCH_REPO_REVISION)
retail_debug_env = tau_bench_runtime.get_environment()
retail_debug_messages: list[dict[str, str]] = [
    {"role": "user", "content": debug_user_message_from_task(DEBUG_RETAIL_TASK)}
]
scripted_user_reply_index = 0

print("Teaching/debug retail run")
print("Task id:", DEBUG_RETAIL_TASK["id"])
print("`tau2` package source checkout:", tau_bench_runtime.source_checkout)
print("Student runtime: MLX-LM")
print("Max agent steps:", DEBUG_AGENT_MAX_STEPS)
print("Note: SIMULATED USER here is a visible notebook helper, not the official τ³-Bench user simulator.")

print_turn("SIMULATED USER / debug task message", retail_debug_messages[-1]["content"])

for agent_step in range(1, DEBUG_AGENT_MAX_STEPS + 1):
    raw_debug_output = generate_student_debug_turn(retail_debug_messages)
    cleaned_debug_output = qwen_format.strip_generated_special_tokens(raw_debug_output)
    retail_debug_messages.append({"role": "assistant", "content": cleaned_debug_output})

    print_turn(f"AGENT / {STUDENT_MODEL} / step {agent_step} raw output", raw_debug_output)

    parsed_debug = qwen_format.parse_qwen_tool_calls(raw_debug_output)
    if parsed_debug.errors:
        print_turn("HARNESS / parser", parsed_debug.errors)

    if not parsed_debug.calls:
        print_turn("AGENT / customer-facing text", cleaned_debug_output)
        if scripted_user_reply_index >= len(DEBUG_USER_FOLLOW_UPS):
            print_turn(
                "HARNESS / stop",
                "The agent did not emit a tool call, and there are no scripted debug user replies left.",
            )
            break

        debug_user_reply = DEBUG_USER_FOLLOW_UPS[scripted_user_reply_index]
        scripted_user_reply_index += 1
        retail_debug_messages.append({"role": "user", "content": debug_user_reply})
        print_turn("SIMULATED USER / scripted debug reply", debug_user_reply)
        continue

    for call_index, call in enumerate(parsed_debug.calls, start=1):
        try:
            coerced_arguments = tau_runtime.coerce_tau_bench_retail_call_arguments(call, retail_tool_schema_by_name)
            print_turn(
                f"HARNESS / parsed tool call {call_index}",
                {"name": call.name, "arguments": coerced_arguments},
            )
            env_result = retail_debug_env.make_tool_call(
                call.name,
                requestor="assistant",
                **coerced_arguments,
            )
            env_result_text = retail_debug_env.to_json_str(env_result)
        except Exception as exc:
            env_result_text = json.dumps(
                {"error_type": type(exc).__name__, "message": str(exc)},
                ensure_ascii=False,
            )

        retail_debug_messages.append({"role": "tool", "content": env_result_text})
        print_turn("ENV / tau2 runtime retail tool result", env_result_text)
else:
    print_turn(
        "HARNESS / stop",
        "Reached DEBUG_AGENT_MAX_STEPS. This guard is only for the interactive teaching cell, not a benchmark score.",
    )


Teaching/debug retail run
Task id: 0
`tau2` package source checkout: /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench
Student runtime: MLX-LM
Max agent steps: 10
Note: SIMULATED USER here is a visible notebook helper, not the official τ³-Bench user simulator.

SIMULATED USER / debug task message
------------------------------------------------------------------------------------------------
You received your order #W2378156 and wish to exchange the mechanical keyboard for the same one but with clicky switches and the smart thermostat for one compatible with Google Home instead of Apple HomeKit. If there is no keyboard that is clicky, RGB backlight, full size, you'd go for no backlight.
You are Yusuf Rossi in zip code 19122.
You do not remember your email address.

AGENT / mlx-community/Qwen3.5-0.8B-MLX-bf16 / step 1 raw output
------------------------------------------------------------------------------------------------
I need to help you with your order 

## Official τ³-Bench Student Baseline Eval

Now we run the actual held-out retail benchmark for the student. This is different from the first-action probe: every task gets a user simulator, a real retail environment, tool execution, a full conversation trajectory, and an evaluator reward.

This is **pass@1**. One trajectory per task. No retry search. The user simulator model is external runtime config loaded by the common helper, and the eval knobs are explicit variables in the next cell.


In [ ]:
if "student_model" not in globals():
    raise RuntimeError("Run the MLX student model loading cell first.")
if "student_sampler" not in globals():
    raise RuntimeError("Run the MLX student model loading cell first, so student_sampler exists.")

# The user simulator is part of the benchmark environment, not the student.
# Its model/API are external runtime config loaded by the common helper.
user_simulator_runtime = user_simulator.start_tau_bench_user_simulator_from_env(
    existing_shim=globals().get("chatgpt_user_simulator_shim")
)
chatgpt_user_simulator_shim = user_simulator_runtime.shim
TAU_BENCH_USER_SIMULATOR_LLM = user_simulator_runtime.model
TAU_BENCH_USER_SIMULATOR_ARGS = user_simulator_runtime.args

TAU_BENCH_STUDENT_EVAL_LIMIT = len(retail_eval_tasks)
TAU_BENCH_STUDENT_EVAL_MAX_STEPS = 100
TAU_BENCH_STUDENT_EVAL_MAX_ERRORS = 10
TAU_BENCH_STUDENT_EVAL_MAX_NEW_TOKENS = MAX_NEW_TOKENS
TAU_BENCH_STUDENT_EVAL_SEED = 42
TAU_BENCH_MLFLOW_ENABLED = True
TAU_BENCH_MLFLOW_EXPERIMENT_NAME = "distillation-blogs-tau3"
TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS = True

user_simulator_slug = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
student_slug = cfg.filename_slug(STUDENT_MODEL)
student_eval_output_path = OUTPUT_DIR / f"{student_slug}_tau3_bench_retail_test_official_student_eval_{user_simulator_slug}.json"
student_eval_trace_dir = OUTPUT_DIR / "local_traces" / f"{student_slug}_tau3_bench_retail_test_official_student_eval_{user_simulator_slug}"
student_eval_mlflow_run_name = f"{student_slug}_tau3_retail_student_eval_{user_simulator_slug}"
student_eval_mlflow_config = cfg.MlflowConfig(
    enabled=TAU_BENCH_MLFLOW_ENABLED,
    experiment_name=TAU_BENCH_MLFLOW_EXPERIMENT_NAME,
    log_full_artifacts=TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS,
    log_spans=False,
)

student_eval_config = cfg.TauBenchRetailEvalConfig(
    dataset_revision=TAU_BENCH_REPO_REVISION,
    student_model_name=STUDENT_MODEL,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
    max_steps=TAU_BENCH_STUDENT_EVAL_MAX_STEPS,
    max_errors=TAU_BENCH_STUDENT_EVAL_MAX_ERRORS,
    max_new_tokens=TAU_BENCH_STUDENT_EVAL_MAX_NEW_TOKENS,
    seed=TAU_BENCH_STUDENT_EVAL_SEED,
    model_role="mlx_student",
)

retail_eval_task_objects = tau_bench_runtime.get_tasks("test")[:TAU_BENCH_STUDENT_EVAL_LIMIT]
student_eval_runner = retail_eval.TauBenchRetailMlxStudentEvalRunner(
    runtime=tau_bench_runtime,
    model=student_model,
    tokenizer=tokenizer,
    qwen_tools=retail_tools,
    tool_schema_by_name=retail_tool_schema_by_name,
    sampler=student_sampler,
    config=student_eval_config,
    trace_dir=student_eval_trace_dir,
)

print("Official τ³-Bench retail student eval")
print("Student model:", STUDENT_MODEL)
print("Student runtime: MLX-LM")
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("ChatGPT subscription shim model:", user_simulator_runtime.shim_model)
print("ChatGPT subscription shim base URL:", chatgpt_user_simulator_shim.base_url)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("User simulator API key present:", bool(TAU_BENCH_USER_SIMULATOR_ARGS.get("api_key")))
print("Held-out retail tasks:", len(retail_eval_task_objects))
print("Task execution: sequential, one benchmark task at a time")
print("Max steps per task:", TAU_BENCH_STUDENT_EVAL_MAX_STEPS)
print("Max tool errors per task:", TAU_BENCH_STUDENT_EVAL_MAX_ERRORS)
print("Max new tokens per agent action:", TAU_BENCH_STUDENT_EVAL_MAX_NEW_TOKENS)
print("Output path:", student_eval_output_path)
print("Trace dir:", student_eval_trace_dir)
print("MLflow enabled:", student_eval_mlflow_config.enabled)
print("MLflow tracking URI:", student_eval_mlflow_config.tracking_uri)
print("MLflow experiment:", student_eval_mlflow_config.experiment_name)
print("MLflow full artifacts:", student_eval_mlflow_config.log_full_artifacts)
print("MLflow run name:", student_eval_mlflow_run_name)
print("`tau2` package source checkout:", tau_bench_runtime.source_checkout)


### Check The ChatGPT Subscription User Simulator On One Retail Task

Before running the full held-out eval, make one real τ³-Bench user-simulator call on one retail test task. This confirms the local ChatGPT subscription shim is running, LiteLLM can call it through the OpenAI-compatible local endpoint, and the τ³-Bench task/user-tool wiring works.

This cell does **not** score the student. It only checks the user simulator provider path.


In [16]:

user_simulator_check_task = retail_eval_task_objects[0]
user_simulator_check = retail_eval.check_tau_bench_retail_user_simulator(
    runtime=tau_bench_runtime,
    task=user_simulator_check_task,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
)

print("User simulator check passed.")
print("Task id:", user_simulator_check["task_id"])
print("Model:", user_simulator_check["model"])
print("User-side tool count:", user_simulator_check["user_side_tool_count"])
if user_simulator_check["content"] and user_simulator_check["content"].strip():
    print("First simulated user message:")
    print(user_simulator_check["content"].strip())
if user_simulator_check["tool_calls"]:
    print("First simulated user tool calls:")
    print(json.dumps(user_simulator_check["tool_calls"], indent=2, ensure_ascii=False))


User simulator check passed.
Task id: 5
Model: openai/gpt-5.4-mini
User-side tool count: 0
First simulated user message:
Hi, I’d like to exchange a water bottle and a desk lamp I bought.


### Run The Held-Out Student Eval

The next cell runs the 40 held-out retail tasks. It saves after every task, so rerunning the cell resumes from cached task IDs instead of starting over.

Notebook output is intentionally compact: a progress bar shows completed tasks, `valid/total`, and running accuracy. The noisy τ³/tau2 turn-by-turn logs are suppressed in the notebook. Full task traces are still saved locally, and when MLflow is enabled the same per-task details are logged as artifacts under one eval run.


In [17]:
student_eval_payload = retail_eval.run_tau_bench_retail_eval_tasks(
    task_objects=retail_eval_task_objects,
    runner=student_eval_runner,
    output_path=student_eval_output_path,
    print_progress=True,
    show_progress_bar=True,
    quiet_tau2_console=True,
    mlflow_config=student_eval_mlflow_config,
    mlflow_run_name=student_eval_mlflow_run_name,
    mlflow_tags={"tau3.notebook": "01_student_eval", "tau3.runtime": "mlx_lm"},
)

student_eval_rows = student_eval_payload["task_results"]
student_eval_correct = sum(1 for row in student_eval_rows if row["is_success"])
student_eval_accuracy = student_eval_correct / len(student_eval_rows) if student_eval_rows else 0.0

print("\nStudent official τ³-Bench retail pass@1 accuracy:", round(student_eval_accuracy, 4))
print(f"Correct: {student_eval_correct}/{len(student_eval_rows)}")
print("Saved aggregate results to:", student_eval_output_path)
print("Saved per-task traces to:", student_eval_trace_dir)
if student_eval_mlflow_config.enabled:
    print("MLflow run:", student_eval_mlflow_run_name)
    print("MLflow experiment:", student_eval_mlflow_config.experiment_name)

failure_counter = Counter(
    row["termination_reason"] if not row.get("error") else f"error:{row['error']['type']}"
    for row in student_eval_rows
    if not row["is_success"]
)
print("\nFailure/termination summary:")
for name, count in failure_counter.most_common():
    print(f"  {name}: {count}")


Cached tasks: 0
Pending tasks: 40


mlx_student eval:   0%|          | 0/40 [00:00<?, ?task/s]

🏃 View run mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_student_eval_openai_gpt_5_4_mini at: http://127.0.0.1:5050/#/experiments/6/runs/c38c620db4db42a685c56ea74ac58ba2
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/6

Student official τ³-Bench retail pass@1 accuracy: 0.2
Correct: 8/40
Saved aggregate results to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_bench_retail_test_official_student_eval_openai_gpt_5_4_mini.json
Saved per-task traces to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/local_traces/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_bench_retail_test_official_student_eval_openai_gpt_5_4_mini
MLflow run: mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_student_eval_openai_gpt_5_4_mini
MLflow experiment: distillation-blogs-tau3

Failure/termination summary:
  user_stop: 19
  error:InternalServerError: 9
  max_steps: 2
  too_many_errors: 2


In [ ]:
# Eval trace statistics: student, teacher, and later the trained student.
# Run this after any eval notebook has produced its JSON file. Missing files are skipped.
from IPython.display import display
from common import trace_stats as trace_stats_module

if "TAU_BENCH_USER_SIMULATOR_LLM" not in globals():
    raise RuntimeError("Run the student eval setup cell first so TAU_BENCH_USER_SIMULATOR_LLM is defined.")

trace_stats_bundle = trace_stats_module.load_tau_bench_eval_trace_stats(
    output_dir=OUTPUT_DIR,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    student_model=STUDENT_MODEL,
)

print("Result files loaded:")
for label, path in trace_stats_bundle["result_files"]:
    print(f"- {label}: {path}")
if not trace_stats_bundle["result_files"]:
    print("No eval result files found yet. Run the student, teacher, or trained-student eval first.")

stats_df = trace_stats_bundle["stats_df"]
tool_df = trace_stats_bundle["tool_df"]
if not stats_df.empty:
    display(trace_stats_bundle["summary_df"].round(3))

    print("Slowest tasks:")
    display(trace_stats_bundle["slowest_df"])

    if not tool_df.empty:
        print("Most-used tools by run:")
        display(trace_stats_bundle["top_tools_df"])

    stats_csv_path = trace_stats_module.save_tau_bench_eval_trace_stats_csv(stats_df, OUTPUT_DIR)
    print("Saved per-task stats to:", stats_csv_path)

    trace_stats_module.plot_tau_bench_eval_trace_stats(stats_df)

## Where We Are Now

Notebook 1 now does the baseline job it is supposed to do:

- The toy harness explains the mechanics.
- The real dataset is τ³-Bench retail.
- The student prompt uses real retail policy and tool schemas.
- The optional first-action probe is only a quick diagnostic.
- The official student eval section runs the held-out retail test tasks with the τ³-Bench user simulator, retail environment, tool execution, and evaluator reward.

The score from this notebook is the baseline student pass@1. Notebook 2 runs the teacher on the same held-out retail split, using the same harness boundary.
